# functionalRhoPCA on the Berkeley Growth Dataset

This notebook uses the Berkeley growth curves from `scikit-fda`, constructs the required long-format `DataFrame`, fits `functionalRhoPCA` with boys as target and girls as background, and plots:

- the first generalized eigenfunction
- the second generalized eigenfunction
- the GE1 vs GE2 score scatter for boys and girls


In [14]:
!pip install -U scikit-fda==0.10.1


ERROR: Ignored the following versions that require a different python version: 0.10.1 Requires-Python >=3.10
ERROR: Could not find a version that satisfies the requirement scikit-fda==0.10.1 (from versions: 0.2, 0.2.1, 0.2.2, 0.2.3, 0.3, 0.4, 0.5, 0.6, 0.6.1, 0.7, 0.7.1, 0.8, 0.8.1, 0.9, 0.9.1)
ERROR: No matching distribution found for scikit-fda==0.10.1

[notice] A new release of pip is available: 23.3.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skfda.datasets import fetch_growth

repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from rhopca.methods import functionalRhoPCA


In [2]:
# --- Load dataset ---
dataset = fetch_growth()
fd = dataset["data"]
y = np.array(dataset["target"])

boys_mask = y == 0
girls_mask = y == 1

boys_fd = fd[boys_mask]
girls_fd = fd[girls_mask]

ages = np.asarray(fd.grid_points[0], dtype=float)
heights = np.asarray(fd.data_matrix[..., 0], dtype=float)

n_subjects = heights.shape[0]
subject_ids = np.arange(n_subjects)
sex_labels = np.where(y == 0, "boy", "girl")

growth_df = pd.DataFrame(
    {
        "child_id": np.repeat(subject_ids, len(ages)),
        "age": np.tile(ages, n_subjects),
        "height": heights.reshape(-1),
        "sex": np.repeat(sex_labels, len(ages)),
    }
)

growth_df.head()


,child_id,age,height,sex
0,0,1.00,81.3,boy
1,0,1.25,84.2,boy
2,0,1.50,86.4,boy
3,0,1.75,88.9,boy
4,0,2.00,91.4,boy


In [3]:
growth_df.groupby("sex")["child_id"].nunique()


sex
boy     39
girl    54
Name: child_id, dtype: int64

In [4]:
mdl = functionalRhoPCA(
    growth_df,
    contrast_column="sex",
    target="boy",
    background="girl",
    observation="child_id",
    time="age",
    gene="height",
    basis="bspline",
    num_basis_functions=7,
    n_components=7,
)

mdl.fit()
mdl.eigvals


ImportError: functionalRhoPCA requires the optional dependency 'scikit-fda'. Install it with `pip install scikit-fda`.

In [7]:
from skfda.representation.irregular import FDataIrregular

ModuleNotFoundError: No module named 'skfda.representation.irregular'

In [9]:
import skfda.representation

In [13]:
# from skfda.representation import irregular
skfda.__version__

'0.9.1'

In [ ]:
observed_time = np.linspace(
    growth_df["age"].min(),
    growth_df["age"].max(),
    200,
    endpoint=True,
)

mdl.plot(plot_type="eigenfunction", components=1, time_values=observed_time)
mdl.plot(plot_type="eigenfunction", components=2, time_values=observed_time)


In [ ]:
mdl.plot(plot_type="scatter", components=(1, 2))


In [ ]:
pd.DataFrame(
    {
        "GE 1": mdl.eigvals[: len(mdl.eigenfunction_dict)],
    },
    index=[f"Component {i}" for i in range(1, len(mdl.eigenfunction_dict) + 1)],
)
